# GMM ve OCSVM ile Medicare Fraud Tespiti

Bu notebook, Medicare Part D veri seti üzerinde GMM ve One-Class SVM modellerini eğitir.

## Modeller:
- **One-Class SVM (OCSVM)**: RBF kernel ile anomali tespiti
- **Gaussian Mixture Model (GMM)**: Olasılıksal anomali tespiti

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import OneClassSVM
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


## 1. Medicare Verisini Yükle

In [ ]:
# Medicare verisini yükle
combined_df = pd.read_csv("../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv")
print(f"Veri seti yüklendi: {combined_df.shape}")
combined_df.head()

Veri seti yüklendi: (798496, 215)


,Tot_Benes_sum,Tot_Benes_mean,Tot_Benes_median,Tot_Benes_std,Tot_Benes_min,Tot_Benes_max,Tot_Clms_sum,Tot_Clms_mean,Tot_Clms_median,Tot_Clms_std,...,"Prscrbr_Type_Technician, Pathology",Prscrbr_Type_Technician/Technologist,Prscrbr_Type_Thoracic Surgery,Prscrbr_Type_Thoracic Surgery (Cardiothoracic Vascular Surgery),Prscrbr_Type_Undefined Physician type,Prscrbr_Type_Unknown Supplier/Provider Specialty,Prscrbr_Type_Urology,Prscrbr_Type_Vascular Surgery,Prscrbr_Type_Veterinarian,Prscrbr_Type_Voluntary Health or Charitable Agency
0,50.0,16.666667,12.0,8.962886,11.0,27.0,58.0,19.333333,14.0,11.930353,...,False,False,False,False,False,False,False,False,False,False
1,655.0,21.833333,16.5,11.462332,11.0,56.0,2578.0,41.580645,25.5,39.281059,...,False,False,False,False,False,False,False,False,False,False
2,0.0,0.000000,0.0,0.000000,0.0,0.0,13.0,13.000000,13.0,0.000000,...,False,False,False,False,False,False,False,False,False,False
3,492.0,27.333333,21.0,16.932392,12.0,66.0,499.0,27.722222,21.0,17.238144,...,False,False,False,False,False,False,False,False,False,False
4,36.0,18.000000,18.0,9.899495,11.0,25.0,210.0,26.250000,22.0,18.583787,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Sınıf dağılımını kontrol et
print("Sınıf dağılımı:")
print(combined_df['TARGET'].value_counts())

Sınıf dağılımı:
TARGET
NOT_FRAUD    797540
FRAUD           956
Name: count, dtype: int64


## 2. Veri Hazırlığı

In [ ]:
# TARGET'ı sayısala çevir
combined_df['TARGET'] = combined_df['TARGET'].map({'NOT_FRAUD': 0, 'FRAUD': 1})

# Feature ve target ayır
X = combined_df.drop(columns=['TARGET'])
y = combined_df['TARGET']

# Boolean sütunları int'e çevir
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

# Sadece numerik sütunları al
X = X.select_dtypes(include=[np.number])

# NaN temizle
X = X.fillna(0)

print(f"X boyutu: {X.shape}")
print(f"Sınıf dağılımı: Normal={sum(y==0)}, Fraud={sum(y==1)}")

X boyutu: (798496, 214)
Sınıf dağılımı: Normal=797540, Fraud=956


In [ ]:
# Büyük veri seti için örnekle
sample_size = min(100000, len(X))
X_sample, _, y_sample, _ = train_test_split(
    X.values, y.values, train_size=sample_size, stratify=y, random_state=42
)

# Normalize
scaler = StandardScaler()
X_sample = scaler.fit_transform(X_sample)

print(f"Örneklenmiş veri boyutu: {X_sample.shape}")
print(f"Fraud sayısı: {sum(y_sample==1)}")

Örneklenmiş veri boyutu: (100000, 214)
Fraud sayısı: 120


## 3. Çapraz Doğrulama Ayarları

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
repeats = 3

# Sonuçları sakla
results_med = {
    'majority': {
        'OCSVM': [],
        'GMM': []
    }
}

print(f"K-Fold: 3, Tekrar: {repeats}")

K-Fold: 3, Tekrar: 3


## 4. Model Eğitimi

In [ ]:
max_train_size = 10000  # Hız için subset

for r in range(repeats):
    print(f"\n--- Medicare: Tekrar {r+1}/{repeats} ---")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + r)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X_sample[train_idx], X_sample[test_idx]
        y_train, y_test = y_sample[train_idx], y_sample[test_idx]
        
        # Majority class (normal) eğitim verisi
        X_train_major = X_train[y_train == 0][:max_train_size]
        
        # -- One-Class SVM --
        ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
        ocsvm.fit(X_train_major)
        ocsvm_scores = -ocsvm.decision_function(X_test)
        
        auc_ocsvm = roc_auc_score(y_test, ocsvm_scores)
        aupr_ocsvm = average_precision_score(y_test, ocsvm_scores)
        results_med['majority']['OCSVM'].append((auc_ocsvm, aupr_ocsvm))
        
        # -- GMM --
        gmm = GaussianMixture(n_components=1, random_state=42)
        gmm.fit(X_train_major)
        gmm_scores = -gmm.score_samples(X_test)
        
        auc_gmm = roc_auc_score(y_test, gmm_scores)
        aupr_gmm = average_precision_score(y_test, gmm_scores)
        results_med['majority']['GMM'].append((auc_gmm, aupr_gmm))
        
        if fold == 1:
            print(f"  Fold 1 - OCSVM: AUC={auc_ocsvm:.4f}, GMM: AUC={auc_gmm:.4f}")

print("\n✓ Medicare eğitimi tamamlandı!")


--- Medicare: Tekrar 1/3 ---
  Fold 1 - OCSVM: AUC=0.5684, GMM: AUC=0.5686

--- Medicare: Tekrar 2/3 ---
  Fold 1 - OCSVM: AUC=0.5684, GMM: AUC=0.5686

--- Medicare: Tekrar 3/3 ---
  Fold 1 - OCSVM: AUC=0.5684, GMM: AUC=0.5686

✓ Medicare eğitimi tamamlandı!


## 5. Değerlendirme

In [ ]:
print("="*60)
print("MEDICARE - SONUÇ KARŞILAŞTIRMASI")
print("="*60)

for group in results_med:
    print(f'\n>>> {group.upper()}')
    print(f"{'Model':20s} | {'Mean AUC':10s} | {'Mean AUPRC':10s}")
    print("-"*45)
    
    for method in results_med[group]:
        scores = results_med[group][method]
        if scores:
            arr = np.array(scores)
            mean_auc, mean_aupr = arr.mean(axis=0)
            print(f'{method:20s} | {mean_auc:.4f}     | {mean_aupr:.4f}')

MEDICARE - SONUÇ KARŞILAŞTIRMASI

>>> MAJORITY
Model                | Mean AUC   | Mean AUPRC
---------------------------------------------
OCSVM                | 0.5326     | 0.0022
GMM                  | 0.5857     | 0.0023


## 6. Sonuçları Kaydet

In [ ]:
os.makedirs("../pickled_storage", exist_ok=True)

with open("../pickled_storage/results_med_gmm_ocsvm.pkl", "wb") as f:
    pickle.dump(results_med, f)

print("✓ Sonuçlar kaydedildi: pickled_storage/results_med_gmm_ocsvm.pkl")

✓ Sonuçlar kaydedildi: pickled_storage/results_med_gmm_ocsvm.pkl
